# 2. Tokenize transcriptomes and data pairing



This notebook starts with the CPU-only tokenization step. Replace placeholder paths like `path/to/...` with real locations on your system.


## 2.1. Configure paths and parameters

These parameters mirror the CLI options in `perturbgen.pp.GF_tokenisation`.


In order to download data, including the LPS data for this tutorial see https://perturbgen.cog.sanger.ac.uk/docs/data.html

In [1]:
# download LPS data from AWS S3 bucket
!aws --endpoint-url https://cog.sanger.ac.uk --no-sign-request \
  s3 cp s3://perturbgen/Manuscript/lps_otar.h5ad ./lps_otar.h5ad

download: s3://perturbgen/Manuscript/lps_otar.h5ad to ./lps_otar.h5ad


In [2]:
from pathlib import Path

REPO_ROOT = Path("/home/stuke1/perturbgen/Perturbgen")

H5AD_PATH = str(REPO_ROOT / "docs/examples/lps_otar.h5ad")
DATASET_NAME = "LPS_all_tps_2k"  # choose a name for the dataset
GENE_FILTERING_MODE = "hvg"  # one of: hvg, degs, all
HVG_MODE = "before_tokenisation"  # before_tokenisation or after_tokenisation

VAR_LIST = [
    "cell_type_harmonized",
    "time_after_LPS",
]  # list of obs to retain in adata.vars after preprocessing

PAIRING_MODE = "stratified"  # stratified, random, mapping
TIME_OBS = "time_after_LPS"  # obs with time point info for pairing
PAIRING_FILE = "path/to/pairing.csv"  # only if PAIRING_MODE == 'mapping'
MAIN_PAIRING_OBS = "cell_type_harmonized"  # main obs for pairing (with TIME_OBS)
OPT_PAIRING_OBS = []  # optional additional obs

NPROC = 8  # number of parallel processes
N_HVG = 2000  # number of HVGs if HVG filtering is used
TIME_POINT_ORDER = ["normal", "90m_LPS", "6h_LPS", "10h_LPS"]
REFERENCE_TIME = "normal"  # source/control time point for pairing

# pretrained Geneformer 95M dicts shipped with the repo
GENE_MEDIAN_PATH = str(REPO_ROOT / "perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl")
TOKEN_DICT_PATH = str(REPO_ROOT / "perturbgen/pp/token_dict_gftokens_gc95M.pkl")
GENE_MAPPING_PATH = str(REPO_ROOT / "perturbgen/pp/ensembl_mapping_dict_gc95M.pkl")


### Choosing `REFERENCE_TIME` (the source state)

`REFERENCE_TIME` sets which time point becomes the **source** state for pairing. The tokenizer writes the source files as `{REFERENCE_TIME}.dataset` / `{REFERENCE_TIME}.h5ad`, and every pair runs *source → later target time points* (per `TIME_POINT_ORDER`).

This LPS tutorial is tokenized **twice**, for two different analyses:

| Analysis | `REFERENCE_TIME` | `DATASET_NAME` | Used by |
|----------|------------------|----------------|---------|
| Cell / gene embeddings, gene programs | `normal` | `LPS_all_tps_2k` | notebooks 04, 05 |
| In silico perturbation (e.g. IL1B knockout) | `90m_LPS` | `lps_90min_perturb` | notebook 06 |

**Why two runs?** For a *source* ("src") knockout, the perturbed gene must be expressed in the source state — otherwise there is nothing to knock out: cells without the gene token in the source are filtered out, and batches with no qualifying cells are **skipped entirely**. IL1B is induced by LPS and is ~0 at the unstimulated `normal` baseline, so the IL1B knockout uses the `90m_LPS` source. Re-run this notebook with `REFERENCE_TIME = "90m_LPS"` and a distinct `DATASET_NAME` (e.g. `lps_90min_perturb`) to produce the source used by notebook 06; keep `REFERENCE_TIME = "normal"` for the embedding analyses.

## 2.2. Build the tokenization command

This prints the exact command that will be executed. Review it before running.


In [5]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "tokenise",
    "--h5ad_path", H5AD_PATH,
    "--dataset", DATASET_NAME,
    "--gene_filtering_mode", GENE_FILTERING_MODE,
    "--hvg_mode", HVG_MODE,
    "--var_list", *VAR_LIST,
    "--pairing_mode", PAIRING_MODE,
    "--time_obs", TIME_OBS,
    "--main_pairing_obs", MAIN_PAIRING_OBS,
    "--nproc", str(NPROC),
    "--n_hvg", str(N_HVG),
    "--reference_time", REFERENCE_TIME,
    "--time_point_order", *TIME_POINT_ORDER,
    "--gene_median_path", GENE_MEDIAN_PATH,
    "--token_dict_path", TOKEN_DICT_PATH,
    "--gene_mapping_path", GENE_MAPPING_PATH,
]

if PAIRING_MODE == "mapping":
    cmd += ["--pairing_file", PAIRING_FILE]
if OPT_PAIRING_OBS:
    cmd += ["--opt_pairing_obs", *OPT_PAIRING_OBS]

print(" ".join(cmd))


python -m perturbgen tokenise --h5ad_path /home/stuke1/perturbgen/Perturbgen/docs/examples/lps_otar.h5ad --dataset LPS_all_tps_2k --gene_filtering_mode hvg --hvg_mode before_tokenisation --var_list cell_type_harmonized time_after_LPS --pairing_mode stratified --time_obs time_after_LPS --main_pairing_obs cell_type_harmonized --nproc 8 --n_hvg 2000 --reference_time normal --time_point_order normal 90m_LPS 6h_LPS 10h_LPS --gene_median_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl --token_dict_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl --gene_mapping_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl


## 2.3. Run tokenization (CPU-only)

This step can take time depending on dataset size.


In [6]:
import subprocess

subprocess.run(cmd, check=True)


loading, please wait...
Current working directory: /home/stuke1/perturbgen
Start preprocessing adata...
Number of genes dropped: 0
Finished preprocessing adata.
Start tokenisation of adata...
Tokenizing /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_GF_genes/LPS_all_tps_2k.h5ad


/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/tokenizer.py:495: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/tokenizer.py:498: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_GF_genes/LPS_all_tps_2k.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Saving the dataset (1/1 shards): 100%|██████████| 223478/223478 [00:00<00:00, 311047.35 examples/s]


Finished tokenisation.


/home/stuke1/perturbgen/Perturbgen/perturbgen/src/utils.py:1502: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_obs_.groupby(grouping_obs)[time_obs].transform('nunique') == total_tps
/home/stuke1/perturbgen/Perturbgen/perturbgen/src/utils.py:1505: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = adata_grouped.groupby(grouping_obs)
  0%|          | 0/4 [00:00<?, ?it/s]/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)

 25%|██▌   

CompletedProcess(args=['python', '-m', 'perturbgen', 'tokenise', '--h5ad_path', '/home/stuke1/perturbgen/Perturbgen/docs/examples/lps_otar.h5ad', '--dataset', 'LPS_all_tps_2k', '--gene_filtering_mode', 'hvg', '--hvg_mode', 'before_tokenisation', '--var_list', 'cell_type_harmonized', 'time_after_LPS', '--pairing_mode', 'stratified', '--time_obs', 'time_after_LPS', '--main_pairing_obs', 'cell_type_harmonized', '--nproc', '8', '--n_hvg', '2000', '--reference_time', 'normal', '--time_point_order', 'normal', '90m_LPS', '6h_LPS', '10h_LPS', '--gene_median_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl', '--token_dict_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl', '--gene_mapping_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl'], returncode=0)

## 2.4. Outputs

Tokenized files are written under the tokenized data directory defined in `perturbgen/configs/paths.py`.
You should see a new folder for your dataset name containing `.dataset` files and pairing outputs in root_dir/T_perturb/tokenized_data.


---
Next: training steps (masking model and count decoder) in the following sections. See Notebook 3 for training the Perturbgen model
